# Solution: Non-Graphical EDA — Swiss EV Charging Stations

**Dataset:** Swiss public EV charging stations (source: [Swiss Federal Office of Energy, OICP open data](https://data.geo.admin.ch/ch.bfe.ladestellen-elektromobilitaet/))

In [1]:
# Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### Task 1 — Load and inspect the dataset

In [2]:
df = pd.read_csv('../data/ev_charging_stations.csv')
print('Shape:', df.shape)
df.head(3)

Shape: (2442, 15)


,evse_id,station_name,street,city,postal_code,lat,lon,power_kw,power_type,amperage,voltage,plug_type,is_open_24h,renewable_energy,auth_mode
0,CH*CCI*E22078,SIG CERN Esplanade des Particules,SIG CERN Esplanade des Particules,Meyrin,1217,46.23432,6.055602,22.0,AC_3_PHASE,32,230,Type 2 Outlet,True,False,NFC RFID Classic
1,CH*CCI*E22076,SIG CERN Esplanade des Particules,SIG CERN Esplanade des Particules,Meyrin,1217,46.23432,6.055602,22.0,AC_3_PHASE,32,230,Type 2 Outlet,True,False,NFC RFID Classic
2,CH*CCI*E22079,SIG CERN Esplanade des Particules,SIG CERN Esplanade des Particules,Meyrin,1217,46.23432,6.055602,22.0,AC_3_PHASE,32,230,Type 2 Outlet,True,False,NFC RFID Classic


In [3]:
print('Data types:')
print(df.dtypes)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')

Data types:
evse_id                 str
station_name            str
street                  str
city                    str
postal_code             str
lat                 float64
lon                 float64
power_kw            float64
power_type              str
amperage              int64
voltage               int64
plug_type               str
is_open_24h            bool
renewable_energy       bool
auth_mode               str
dtype: object

Missing values per column:
evse_id             0
station_name        0
street              0
city                0
postal_code         0
lat                 0
lon                 0
power_kw            0
power_type          0
amperage            0
voltage             0
plug_type           0
is_open_24h         0
renewable_energy    0
auth_mode           0
dtype: int64

Duplicate rows: 0


### Task 2 — Mean, median and mode of `power_kw`

In [4]:
print(f"Mean:   {df['power_kw'].mean():.2f} kW")
print(f"Median: {df['power_kw'].median():.2f} kW")
print(f"Mode:   {df['power_kw'].mode()[0]:.2f} kW")
print()
print('Note: mean (50.5 kW) >> median (22 kW) → strongly right-skewed.')
print('Most stations deliver 22 kW (AC), but a few fast-chargers (150–400 kW DC) pull the mean up.')

Mean:   50.50 kW
Median: 22.00 kW
Mode:   22.00 kW

Note: mean (50.5 kW) >> median (22 kW) → strongly right-skewed.
Most stations deliver 22 kW (AC), but a few fast-chargers (150–400 kW DC) pull the mean up.


### Task 3 — Quantiles for numerical columns

In [5]:
num_cols = ['power_kw', 'amperage', 'voltage']
df[num_cols].quantile([0.05, 0.25, 0.50, 0.75, 0.95]).round(1)

,power_kw,amperage,voltage
0.05,11.0,16.0,230.0
0.25,22.0,32.0,230.0
0.50,22.0,32.0,230.0
0.75,22.0,50.0,230.0
0.95,225.0,563.0,400.0


### Task 4 — Variance and standard deviation

In [6]:
print('Sample variance:')
print(df[num_cols].var().round(2))
print()
print('Sample standard deviation:')
print(df[num_cols].std().round(2))
print()
print('The high std of power_kw (77.3 kW) reflects the wide mix of slow AC and fast DC chargers.')

Sample variance:
power_kw     5976.94
amperage    39833.08
voltage      5825.93
dtype: float64

Sample standard deviation:
power_kw     77.31
amperage    199.58
voltage      76.33
dtype: float64

The high std of power_kw (77.3 kW) reflects the wide mix of slow AC and fast DC chargers.


### Task 5 — Skewness and kurtosis of `power_kw`

In [7]:
skew = df['power_kw'].skew()
kurt = df['power_kw'].kurtosis()
print(f'Skewness: {skew:.3f}')
print(f'Excess kurtosis: {kurt:.3f}')

Skewness: 3.002
Excess kurtosis: 8.947


In [8]:
# Interpretation
print('Interpretation:')
print(f'  Skewness = {skew:.2f} → strongly right-skewed (positive).')
print('  Most charging stations provide 22 kW, but a few provide up to 400 kW,')
print('  creating a long tail to the right.')
print()
print(f'  Excess kurtosis = {kurt:.2f} → heavy-tailed (leptokurtic) distribution.')
print('  The extreme values (fast DC chargers) are more frequent than a normal distribution would predict.')

Interpretation:
  Skewness = 3.00 → strongly right-skewed (positive).
  Most charging stations provide 22 kW, but a few provide up to 400 kW,
  creating a long tail to the right.

  Excess kurtosis = 8.95 → heavy-tailed (leptokurtic) distribution.
  The extreme values (fast DC chargers) are more frequent than a normal distribution would predict.


### Task 6 — Contingency table: `power_type` × `plug_type`

In [9]:
ct = pd.crosstab(
    df['power_type'],
    df['plug_type'],
    margins=True,
    margins_name='Total'
)
ct

plug_type,CCS Combo 2 Plug (Cable Attached),CHAdeMO,Type 2 Connector (Cable Attached),Type 2 Outlet,Total
power_type,,,,,
AC_1_PHASE,0,0,0,1,1
AC_3_PHASE,0,0,37,1784,1821
DC,494,125,1,0,620
Total,494,125,38,1785,2442


In [10]:
print('Observation: DC stations exclusively use CCS Combo 2 or CHAdeMO plugs.')
print('AC 3-phase stations almost exclusively use Type 2 Outlet connectors.')
print('This reflects the technical standards: DC fast-charging uses CCS/CHAdeMO, AC uses Type 2.')

Observation: DC stations exclusively use CCS Combo 2 or CHAdeMO plugs.
AC 3-phase stations almost exclusively use Type 2 Outlet connectors.
This reflects the technical standards: DC fast-charging uses CCS/CHAdeMO, AC uses Type 2.


### Task 7 — Pearson correlation matrix

In [11]:
corr = df[num_cols].corr(method='pearson').round(3)
corr

,power_kw,amperage,voltage
power_kw,1.000,0.997,0.672
amperage,0.997,1.000,0.680
voltage,0.672,0.680,1.000


In [12]:
print('Interpretation:')
print('  power_kw ↔ amperage: strong positive correlation.')
print('  Higher charging power requires higher current (P = U × I).')
print()
print('  power_kw ↔ voltage: moderate positive correlation.')
print('  DC fast-chargers use higher voltages (400–800 V) than AC stations (230–400 V).')
print()
print('  amperage ↔ voltage: weaker correlation — both contribute to power independently.')

Interpretation:
  power_kw ↔ amperage: strong positive correlation.
  Higher charging power requires higher current (P = U × I).

  power_kw ↔ voltage: moderate positive correlation.
  DC fast-chargers use higher voltages (400–800 V) than AC stations (230–400 V).

  amperage ↔ voltage: weaker correlation — both contribute to power independently.


## Jupyter notebook --footer info--
(please always provide this at the end of each notebook)

In [13]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('------------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('------------------------------------')

------------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-01 07:37:33
Python Version: 3.13.5
------------------------------------
